# GSE171213 — Harmony batch entegrasyonu (43.438 hücre)
**Amaç:** Önceden oluşturulan 30 boyutlu transkriptomik temsil üzerinde, 12 örneğin kimliğini batch değişkeni olarak kullanarak Harmony çalıştırmak; düzeltme öncesi/sonrası örnek-küme ilişkisini ve fibroblast adaylarının korunmasını değerlendirmek.

**Sınır:** Bu çalışma yalnızca mevcut 30 boyutlu latent uzayı düzeltir; ham sayımlardan yeni HVG/PCA oluşturmaz. Örnek kimliği ile biyolojik durum ilişkili olduğundan daha iyi karışım, tek başına daha iyi biyolojik doğruluk demek değildir. Fibroblast referans anotasyonu bu defterin kapsamında değildir.

**1.** Bu defteri Colab’da açın. Aşağıdaki hücreyi çalıştırın. İstendiğinde `GSE171213_Harmony_Colab_Girdi.zip` dosyasını seçin.

In [ ]:
%pip -q install harmonypy scanpy scikit-learn
from google.colab import files
import io, zipfile, numpy as np, pandas as pd
uploaded = files.upload()
assert 'GSE171213_Harmony_Colab_Girdi.zip' in uploaded, 'Lütfen verilen ZIP dosyasını yükleyin.'
with zipfile.ZipFile(io.BytesIO(uploaded['GSE171213_Harmony_Colab_Girdi.zip'])) as z:
    X = np.load(io.BytesIO(z.read('GSE171213_Harmony_Colab_Embedding.npy')))
    meta = pd.read_csv(io.BytesIO(z.read('GSE171213_Harmony_Colab_Metadata.csv')))
assert X.shape == (43438, 30) and len(meta) == 43438
assert not meta[['ornek','CellIndex']].duplicated().any()
assert meta.ornek.nunique() == 12
print('Girdi doğrulandı:', X.shape, 'örnek:', meta.ornek.nunique())

**2.** Harmony entegrasyonu ve düzeltme sonrası yeniden kümelenme. İşlem birkaç dakika sürebilir.

In [ ]:
import harmonypy as hm
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.stats import chi2_contingency
import time
t0=time.time()
# Harmony expects cells x PCs, and metadata in the same row order.
ho = hm.run_harmony(X, meta, vars_use=['ornek'], random_state=42, max_iter_harmony=20)
Z = np.asarray(ho.Z_corr)
if Z.shape == (30, 43438): Z = Z.T
assert Z.shape == X.shape and np.isfinite(Z).all(), f'Harmony boyut hatası: {Z.shape}'
# Önceki analiz 24 küme kullandığı için aynı k değeriyle karşılaştırıyoruz.
new = KMeans(n_clusters=24, random_state=42, n_init=10).fit_predict(Z)
meta['harmony_kume24'] = new
print('Harmony bitti. Süre (sn):',round(time.time()-t0,1))

**3.** Batch metrikleri ve fibroblast adaylarının kümelerdeki dağılımı. Karışım iyileşmesi tek başına biyolojik sinyalin korunduğunu kanıtlamaz.

In [ ]:
from sklearn.metrics import silhouette_score
def cramer_v(labels, sample):
    tab=pd.crosstab(sample,labels)
    chi2=chi2_contingency(tab,correction=False)[0]
    n=tab.to_numpy().sum()
    return float(np.sqrt((chi2/n)/min(tab.shape[0]-1,tab.shape[1]-1)))
old=meta['transkriptom_kume'].to_numpy()
summary=pd.DataFrame([
    {'durum':'once','cramer_v':cramer_v(old,meta.ornek),'tek_ornek_cogunluk_kume':int((pd.crosstab(old,meta.ornek).div(pd.crosstab(old,meta.ornek).sum(axis=1),axis=0).max(axis=1)>0.5).sum())},
    {'durum':'harmony_sonra','cramer_v':cramer_v(new,meta.ornek),'tek_ornek_cogunluk_kume':int((pd.crosstab(new,meta.ornek).div(pd.crosstab(new,meta.ornek).sum(axis=1),axis=0).max(axis=1)>0.5).sum())}
])
print(summary.to_string(index=False))
print('Eski/yeni küme ARI:',round(adjusted_rand_score(old,new),4))
print('Eski/yeni küme NMI:',round(normalized_mutual_info_score(old,new),4))
fb=meta['fibro_high_confidence_candidate'].astype(str).str.lower().isin(['true','1'])
fibro=pd.crosstab(meta['harmony_kume24'],fb).rename(columns={False:'diger',True:'onceki_fibro_adayi'})
fibro['aday_orani']=fibro['onceki_fibro_adayi']/fibro.sum(axis=1)
print('Fibroblast adaylarının yeni kümelerde dağılımı:')
print(fibro.sort_values('onceki_fibro_adayi',ascending=False).head(12).to_string())
# Görsel kalite kontrol: iki boyutlu UMAP, örnek ve küme renkleri
adata=sc.AnnData(X=Z.astype('float32'),obs=meta[['ornek','harmony_kume24']].copy())
adata.obs['harmony_kume24']=adata.obs['harmony_kume24'].astype('category')
sc.pp.neighbors(adata,n_neighbors=20,use_rep='X',random_state=42)
sc.tl.umap(adata,random_state=42)
sc.pl.umap(adata,color=['ornek','harmony_kume24'],show=False)
import matplotlib.pyplot as plt
plt.savefig('GSE171213_Harmony_UMAP.png',dpi=300,bbox_inches='tight')
plt.show()

**4.** Sonuçları indirip ChatGPT’ye geri yükleyin. Bu dosyalar entegrasyonun değerlendirilmesi içindir; bağımsız referans hücre tipi anotasyonu ayrıca yapılacaktır.

In [ ]:
import json,zipfile
meta.to_csv('GSE171213_Harmony_Hucre_Etiketleri.csv',index=False)
summary.to_csv('GSE171213_Harmony_Batch_Metrikleri.csv',index=False)
fibro.to_csv('GSE171213_Harmony_Fibroblast_Kume_Dagilimi.csv')
np.save('GSE171213_Harmony_Duzeltilmis_Latent.npy',Z.astype('float32'))
with zipfile.ZipFile('GSE171213_Harmony_Sonuc_Paketi.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in ['GSE171213_Harmony_Hucre_Etiketleri.csv','GSE171213_Harmony_Batch_Metrikleri.csv','GSE171213_Harmony_Fibroblast_Kume_Dagilimi.csv','GSE171213_Harmony_Duzeltilmis_Latent.npy','GSE171213_Harmony_UMAP.png']:
        z.write(p)
files.download('GSE171213_Harmony_Sonuc_Paketi.zip')